# Ensemble Visualization

Interactive notebook for visually inspecting ensemble outputs during testing.  
Edit the **Parameters** cell, then run all cells top to bottom.

**What you'll see:**
- Row 1: input density per timestamp
- Row 2: UDM ranked masks per timestamp
- Row 3 (if height band present): input height per timestamp
- Summary: pre-mask ensemble | water mask | elevation mask | final density
- Height summary: input heights + final ensembled height (if height band present)
- Difference view: what changed after masking

**How it works:**  
Calls `process_scene` from `scripts/ensemble.py` directly — no duplicated ensemble logic.


In [ ]:
import os, sys, tempfile
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import transform_bounds
from shapely.geometry import mapping, box

from tempo.postprocess import create_ranked_mask, downsample_ranked_mask
from scripts.ensemble import (
    process_scene,
    _INTERNAL_LABEL_TEMPLATE,
    _internal_udm_url,
)

In [ ]:
# ---------------------------------------------------------------------------
# Parameters — edit these
# ---------------------------------------------------------------------------
SCENE      = "L15-1022E-1367N"
MODEL      = "12-39-p3"
TIMESTAMPS = ["2023q3", "2023q4", "2024q1", "2024q2"]

OUTPUT_SAS = os.environ["OUTPUT_SAS"]
UDM_SAS    = os.environ["UDM_SAS"]
BLOB_SAS   = os.environ.get("BLOB_SAS")  # for pre-computed water/elevation rasters

# Ensemble settings (match what you pass to ensemble.py)
CONFIDENCE_THRESHOLD   = 95
AVERAGING_ALGORITHM    = "mean"     # mean | max | min | mode
SMALL_VALUE_THRESHOLD  = 2 / 255
SMALL_HEIGHT_THRESHOLD = 0.024
CLARITY_THRESHOLD      = 3.5
DEFAULT_BEHAVIOR       = "median"   # median | mean | max | min
ELEVATION_THRESHOLD    = 5100

## Load inputs

In [ ]:
labels  = {}   # timestamp → (512, 512) float32 density array
heights = {}   # timestamp → (512, 512) float32 height array, if band 2 exists
udms    = {}   # timestamp → (H, W) uint8 ranked mask
profile = None
aoi     = None

for i, ts in enumerate(TIMESTAMPS):
    label_url = _INTERNAL_LABEL_TEMPLATE.format(
        scene=SCENE, timestamp=ts, model=MODEL, sas=OUTPUT_SAS
    )
    with rasterio.open(label_url) as src:
        labels[ts] = src.read(1).astype(np.float32)
        if src.count >= 2:
            heights[ts] = src.read(2).astype(np.float32)
        if i == 0:
            profile = src.profile.copy()
            bounds  = transform_bounds(src.crs, "EPSG:4326", *src.bounds)
            aoi     = mapping(box(*bounds))

    udm_url = _internal_udm_url(SCENE, ts, UDM_SAS)
    if udm_url:
        with rasterio.open(udm_url) as src:
            udm = src.read()
        # Pass timestamp so v1/v2 band interpretation is selected correctly.
        udms[ts] = create_ranked_mask(
            udm, confidence_threshold=CONFIDENCE_THRESHOLD, timestamp=ts
        )
    else:
        udms[ts] = None
        print(f"  {ts}: no UDM available")

has_height = len(heights) == len(TIMESTAMPS)
print(f"Loaded {len(labels)} label TIFFs  |  height band: {has_height}")

## Visualize inputs

In [ ]:
n = len(TIMESTAMPS)
nrows = 3 if has_height else 2
fig, axes = plt.subplots(nrows, n, figsize=(5 * n, 5 * nrows))

# Row 0: density per timestamp
for j, ts in enumerate(TIMESTAMPS):
    ax = axes[0, j]
    im = ax.imshow(labels[ts], cmap="viridis", vmin=0, vmax=1)
    ax.set_title(f"Density — {ts}")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Row 1: UDM ranked mask per timestamp
for j, ts in enumerate(TIMESTAMPS):
    ax = axes[1, j]
    mask = udms[ts] if udms[ts] is not None else np.zeros_like(labels[ts])
    im = ax.imshow(mask, cmap="RdYlGn", vmin=1, vmax=4)
    ax.set_title(f"UDM rank — {ts}")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Row 2 (optional): height per timestamp
if has_height:
    vmax_h = max(h.max() for h in heights.values())
    for j, ts in enumerate(TIMESTAMPS):
        ax = axes[2, j]
        im = ax.imshow(heights[ts], cmap="plasma", vmin=0, vmax=vmax_h)
        ax.set_title(f"Height — {ts}")
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(f"{SCENE}  |  {MODEL}", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Run ensemble via `process_scene`

Calls `process_scene` twice — once without masks (pre-mask baseline) and once with water + elevation masks — then reads both output TIFFs for comparison.

In [ ]:
tmpdir = Path(tempfile.mkdtemp())
fp_premask = tmpdir / f"{SCENE}_premask.tif"
fp_final   = tmpdir / f"{SCENE}_final.tif"

print("Running ensemble (no masks)…")
process_scene(
    scene=SCENE, timestamps=TIMESTAMPS, model=MODEL,
    save_fp=fp_premask, output_sas=OUTPUT_SAS, udm_sas=UDM_SAS, blob_sas=BLOB_SAS,
    water=False, elevation=False,
)

print("Running ensemble (with water + elevation masks)…")
process_scene(
    scene=SCENE, timestamps=TIMESTAMPS, model=MODEL,
    save_fp=fp_final, output_sas=OUTPUT_SAS, udm_sas=UDM_SAS, blob_sas=BLOB_SAS,
    water=True, elevation=True,
)

# Read both outputs back for visualization.
with rasterio.open(fp_premask) as src:
    result_pre = src.read().astype(np.float32)   # shape: (bands, H, W)

with rasterio.open(fp_final) as src:
    result_final = src.read().astype(np.float32)

result_pre_mask_density = result_pre[0]
result_final_density    = result_final[0]
has_height_out = result_pre.shape[0] >= 2

if has_height_out:
    result_pre_mask_height = result_pre[1]
    result_final_height    = result_final[1]

# Derive masks from what changed between pre-mask and final outputs.
water_mask     = result_final_density == -1
elevation_mask = (result_pre_mask_density > 0) & (result_final_density == 0)

non_water = result_final_density != -1
print(f"\nDensity — mean (non-water): {result_final_density[non_water].mean():.4f}"
      f"  |  building px: {(result_final_density > 0).sum():,}"
      f"  |  water px: {water_mask.sum():,}")
if has_height_out:
    print(f"Height  — mean (non-water): {result_final_height[non_water].mean():.4f}"
          f"  |  building px: {(result_final_height > 0).sum():,}")

## Summary visualisation — density

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 11))

# --- Row 0: one input density per timestamp ---
for j, ts in enumerate(TIMESTAMPS):
    ax = axes[0, j]
    im = ax.imshow(labels[ts], cmap="viridis", vmin=0, vmax=1)
    ax.set_title(f"Input density — {ts}")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# --- Row 1: ensemble pre-mask | water | elevation | final ---
result_display = np.clip(result_final_density, 0, 1)
mean_final = result_final_density[result_final_density >= 0].mean()
panels = [
    (result_pre_mask_density,      "Ensemble (pre-mask)",                           "viridis",  0, 1),
    (water_mask.astype(float),     f"Water mask ({water_mask.sum():,} px → -1)",    "Blues",    0, 1),
    (elevation_mask.astype(float), f"Elev mask ({elevation_mask.sum():,} px → 0)",  "Oranges",  0, 1),
    (result_display,               f"Final density (mean {mean_final:.4f})",         "viridis",  0, 1),
]
for ax, (data, title, cmap, vmin, vmax) in zip(axes[1], panels):
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(f"{SCENE}  |  {MODEL}  |  {TIMESTAMPS[0]}–{TIMESTAMPS[-1]}", fontsize=14)
plt.tight_layout()
plt.show()

## Summary visualisation — height (if available)

In [ ]:
if has_height_out:
    n = len(TIMESTAMPS)
    fig, axes = plt.subplots(1, n + 1, figsize=(5 * (n + 1), 5))

    vmax_h = max(h.max() for h in heights.values())
    vmax_h = max(vmax_h, result_final_height[result_final_height >= 0].max() if (result_final_height >= 0).any() else vmax_h)

    for j, ts in enumerate(TIMESTAMPS):
        ax = axes[j]
        im = ax.imshow(heights[ts], cmap="plasma", vmin=0, vmax=vmax_h)
        ax.set_title(f"Input height — {ts}")
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax = axes[n]
    result_height_display = np.clip(result_final_height, 0, vmax_h)
    mean_h = result_final_height[result_final_height >= 0].mean()
    im = ax.imshow(result_height_display, cmap="plasma", vmin=0, vmax=vmax_h)
    ax.set_title(f"Final height  (mean {mean_h:.4f})")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.suptitle(f"{SCENE}  |  {MODEL}  |  height band", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No height band in output — skipping height visualisation.")

## Difference view — what changed after masking?

In [ ]:
# diff: pixels that changed value (water=-1 or elevation=0) after masking
diff = result_pre_mask_density - np.clip(result_final_density, 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im0 = axes[0].imshow(result_pre_mask_density, cmap="viridis", vmin=0, vmax=1)
axes[0].set_title("Pre-mask density")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(diff, cmap="hot", vmin=0, vmax=1)
axes[1].set_title(f"Pixels changed by masks ({(diff > 0).sum():,} px)")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(np.clip(result_final_density, 0, 1), cmap="viridis", vmin=0, vmax=1)
axes[2].set_title("Final density (water clipped to 0 for display)")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()